核心思想------开发模式的改变:变换

对TensorIR一个入门程序，对Matmul做优化。(思想还是对内存的复用,类似于gpu的shared mem)

但是是通过对naive版本代码进行一系列显式的变换来看一看整个代码优化的过程

其中会有一系列的tvm中对算子横向 纵向优化(优化程序)的API使用以及最后的结果

主要来自 https://mlc.ai/zh/chapter_tensor_program/case_study.html 中的2.4章节

In [1]:
import numpy as np
import tvm
from tvm.ir.module import IRModule
from tvm.script import tir as T

In [2]:
def lnumpy_mm_relu(A: np.ndarray, B: np.ndarray, C: np.ndarray):
    Y = np.empty((128, 128), dtype="float32")
    for i in range(128):
        for j in range(128):
            for k in range(128):
                if k == 0:
                    Y[i, j] = 0
                Y[i, j] = Y[i, j] + A[i, k] * B[k, j]
    for i in range(128):
        for j in range(128):
            C[i, j] = max(Y[i, j], 0)

上面的程序是实现 mm_relu 操作的一种方式。该程序包含两个阶段：首先我们分配一个中间存储 
，将矩阵乘法的结果存储在那里。然后我们在第二个 for 循环序列中计算 ReLU。你可能会注意到，这肯定不是实现 mm_relu 的唯一方法，当然这可能也不是你想到的第一件事。

无论如何，这确实是实现 mm_relu 的方式之一。我们可以通过将我们的结果与使用数组计算的原始结果进行比较来验证代码的正确性。我们将在本节的后面部分回到这里，并重新讨论其他可能的方法。

In [ ]:
dtype = "float32"
a_np = np.random.rand(128, 128).astype(dtype)
b_np = np.random.rand(128, 128).astype(dtype)

c_mm_relu =  np.maximum(a_np @ b_np, 0)
c_np = np.empty((128, 128), dtype=dtype)

lnumpy_mm_relu(a_np, b_np, c_np)
np.testing.assert_allclose(c_mm_relu, c_np, rtol=1e-5)

下面的代码块展示了 mm_relu 的 TensorIR 实现。这里的代码是用一种名为 TVMScript 的语言实现的，它是一种嵌入在 Python AST 中的特定领域变体。

In [7]:
@tvm.script.ir_module
class MyModule:
    @T.prim_func
    def mm_relu(A: T.Buffer((128, 128), "float32"),
                B: T.Buffer((128, 128), "float32"),
                C: T.Buffer((128, 128), "float32")):
        T.func_attr({"global_symbol": "mmm_relu", "tir.noalias": True})
        Y = T.alloc_buffer((128, 128), dtype="float32")
        for i, j, k in T.grid(128, 128, 128):
            with T.block("Y"):
                vi = T.axis.spatial(128, i)
                vj = T.axis.spatial(128, j)
                vk = T.axis.reduce(128, k)

                # SSR means the properties of each axes are "spatial", "spatial", "reduce"
                # vi, vj, vk = T.axis.remap("SSR", [i, j, k])

                with T.init():
                    Y[vi, vj] = T.float32(0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        # for i, j in T.grid(128, 128):
        #     with T.block("C"):
        #         vi = T.axis.spatial(128, i)
        #         vj = T.axis.spatial(128, j)
        #         C[vi, vj] = T.max(Y[vi, vj], T.float32(0))


    #同样的可以包含多个张量函数:
    @T.prim_func
    def relu(A: T.Buffer((128, 128), "float32"),
             B: T.Buffer((128, 128), "float32")):
        T.func_attr({"global_symbol": "relu", "tir.noalias": True})
        for i, j in T.grid(128, 128):
            with T.block("B"):
                vi, vj = T.axis.remap("SS", [i, j])
                B[vi, vj] = T.max(A[vi, vj], T.float32(0))

个人认为值得注意的主要是vi,vj,vk这三个变量对应的语法堂

vi = T.axis.spatial(128, i)

vj = T.axis.spatial(128, j)

vk = T.axis.reduce(128, k)

上面三行声明了关于块轴的关键性质，语法如下。

[block_axis] = T.axis.[axis_type]([axis_range], [mapped_value])

这三行包含以下信息：

定义了 vi、vj、vk 应被绑定到的位置（在本例中为 i、j 和 k）；

声明了 vi、vj、vk 的原始范围（T.axis.spatial(128, i) 中的 128）；

声明了块轴的属性（spatial, reduce）。

我们一一浏览这些属性。首先，就边界关系而言，vi = T.axis.spatial(128, i) 有效地蕴含了 vi = i。[axis_range] 值提供了 [block_axis] 的预期范围。例如，vi = T.axis.spatial(128, i) 中的 128 表明 vi 应该在 range(0, 128) 中。

In [ ]:
接下来是想要使用变换的形式对TensorIR来进行变换到我们想要的形式

In [8]:
def lnumpy_mm_relu_v2(A: np.ndarray, B: np.ndarray, C: np.ndarray):
    Y = np.empty((128, 128), dtype="float32")
    for i in range(128):
        for j0 in range(32):
            for k in range(128):
                for j1 in range(4):
                    j = j0 * 4 + j1
                    if k == 0:
                        Y[i, j] = 0
                    Y[i, j] = Y[i, j] + A[i, k] * B[k, j]
    for i in range(128):
        for j in range(128):
            C[i, j] = max(Y[i, j], 0)

c_np = np.empty((128, 128), dtype=dtype)
lnumpy_mm_relu_v2(a_np, b_np, c_np)
np.testing.assert_allclose(c_mm_relu, c_np, rtol=1e-5)

NameError: name 'c_mm_relu' is not defined